# 02 — Route gallery: decoded routes vs. real Arctic voyages

**The dissociation this figure visualizes.** On held-out (vessel-disjoint) test
voyages, PEMIRL's context-conditioned policy wins the *per-decision* metric —
its test log-likelihood is ~10.5% better than pooled MCE-IRL (see
`results/reference/results.json`). But per-decision ranking and full-route
decoding are different games:

* **MCE-IRL** (pooled linear reward + finite-horizon soft value iteration)
  decodes routes that stay close to the real corridor (reference test
  Hausdorff ≈ 46 km) — the goal-conditioned soft-VI plan gives it global
  route structure.
* **PEMIRL**'s own generator decodes poorly (reference test Hausdorff
  ≈ 246 km, length ratio ≈ 14): each step is sampled from a reactive policy,
  and small per-step errors compound into long wandering detours.
* **The transfer agent** — a *fresh* action-masked PPO policy trained from
  scratch with the **frozen PEMIRL reward** $f(s,a,z)$ as its only learning
  signal (script `08_reward_transfer.py`) — is the proof that the learned
  reward is *usable*: if a new agent optimizing that reward learns sane,
  goal-reaching navigation, the reward encodes real preferences rather than
  just locally ranking expert moves.

This notebook decodes routes with all three methods on six showcase test
origin–destination pairs and overlays them on the real AIS voyage — the
paper's "money figure". Conditioning is legitimate meta-test protocol: PEMIRL
(and the transfer agent) receive a context $z$ inferred from a small support
set of *other* voyages by the same vessel; the showcased voyage itself is a
query episode, never in the support set.

In [ ]:
import os, sys, time, warnings

# kernel cwd is notebooks/ -> hop to the repo root so config paths resolve
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
REPO = os.getcwd()
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "scripts"))

import arctic_meta_irl.data  # noqa: F401  (must precede env/features/algos imports)
import common                # scripts/common.py — shared pipeline loaders

import numpy as np
import pandas as pd
import torch

from arctic_meta_irl.utils.config import load_config
from arctic_meta_irl.utils.seeding import set_seed
from arctic_meta_irl.algos.mce_irl import MCEIRL
from arctic_meta_irl.algos.pemirl import PEMIRL
from arctic_meta_irl.models.policy import MaskedPolicy
from arctic_meta_irl.env.gcrl_env import GCRLNavEnv, Task
from arctic_meta_irl.eval.rollout import (generate_route_pemirl,
                                          generate_route_policy,
                                          pemirl_support_sets,
                                          _episode_tensors)
from arctic_meta_irl.eval.metrics import route_metrics, route_length_km

warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = "cpu"
cfg = load_config("configs/pemirl.yaml")
set_seed(int(cfg["seed"]))
os.makedirs("notebooks/figures", exist_ok=True)
print("repo root:", REPO)

## Load the cached pipeline and the three released checkpoints

Identical state to `scripts/06_evaluate.py --released`: cached MDP
(`data/cache/mdp_r6.npz`, 14,206 H3 res-6 cells), episodes, vessel-level
splits, and the fitted `FeatureBuilder` (D=20). All models run on CPU.

In [ ]:
t0 = time.time()
mdp, fb, eps = common.load_pipeline(cfg)
env = GCRLNavEnv(mdp, fb, max_horizon=int(cfg["mdp"]["max_horizon"]))

# MCE-IRL: pooled linear reward theta (soft-VI decode)
mce = MCEIRL(mdp, fb, seed=int(cfg["seed"]))
mce.load("models/mce_irl/theta.npz")

# PEMIRL: context-conditioned AIRL (bi-LSTM posterior + masked generator)
pem = PEMIRL(fb.dim, mdp.n_actions, cfg["pemirl"], device=DEVICE)
pem.load("models/pemirl/pemirl.pt")
pem.eval()

# Transfer agent: fresh PPO policy trained on the frozen PEMIRL reward (08)
transfer = MaskedPolicy(fb.dim, mdp.n_actions,
                        context_dim=int(cfg["pemirl"]["context_dim"]),
                        hidden=tuple(cfg["pemirl"]["policy_hidden"]))
transfer.load_state_dict(torch.load("models/ppo_on_pemirl/policy.pt",
                                    map_location=DEVICE))
transfer.to(DEVICE).eval()

print(f"pipeline + 3 checkpoints loaded in {time.time()-t0:.0f}s | "
      f"S={mdp.n_states}, A={mdp.n_actions}, D={fb.dim}, "
      f"test episodes={len(eps['test'])}")

## Showcase O–D pair selection

Same support/query protocol as the paper evaluation
(`pemirl_support_sets(test_eps, support_size=3, seed=0)`): each test vessel's
episodes are split into a small support set (used only to infer $z$) and query
episodes. We select **6 query episodes** as showcase pairs — two each from the
short / medium / long terciles of real-route length, chosen for geographic
spread (distinct vessels, well-separated start points), so the gallery covers
short shuttle runs and multi-week transits across different Arctic regions.

In [ ]:
SUPPORT_SIZE = int(cfg["pemirl"]["support_set_size"])   # 3, as in 06_evaluate
support_map = pemirl_support_sets(eps["test"], SUPPORT_SIZE,
                                  seed=int(cfg["seed"]))

cands = []
for mmsi, (sup, qry) in support_map.items():
    for ep in qry:
        if len(ep) < 8:            # skip trivial hops
            continue
        cands.append((ep, route_length_km(mdp, list(ep.states))))
cands.sort(key=lambda t: t[1])
print(f"{len(cands)} candidate query episodes from "
      f"{len(support_map)} test vessels")

# terciles of real-route length; per tercile pick (a) the median-length
# episode, (b) the episode whose start is farthest from it (distinct vessels)
n = len(cands)
buckets = [cands[:n // 3], cands[n // 3: 2 * n // 3], cands[2 * n // 3:]]
showcase, used_mmsi = [], set()
for bucket in buckets:
    pool = [(ep, km) for ep, km in bucket if ep.mmsi not in used_mmsi]
    first = pool[len(pool) // 2]
    used_mmsi.add(first[0].mmsi)
    la0, lo0 = mdp.latlng[int(first[0].states[0])]
    second, best_d = None, -1.0
    for ep, km in pool:
        if ep.mmsi in used_mmsi:
            continue
        la, lo = mdp.latlng[int(ep.states[0])]
        d = (la - la0) ** 2 + (lo - lo0) ** 2
        if d > best_d:
            second, best_d = (ep, km), d
    used_mmsi.add(second[0].mmsi)
    showcase += [first, second]
showcase.sort(key=lambda t: t[1])

for i, (ep, km) in enumerate(showcase):
    la, lo = mdp.latlng[int(ep.states[0])]
    print(f"pair {i}: mmsi {ep.mmsi:>9d} {ep.category:>6s} "
          f"{ep.year}-{ep.month:02d} | {len(ep):3d} steps, {km:6.0f} km real | "
          f"start ({la:.1f}N, {lo:.1f}W)")

## Decode routes with all three methods

Decoding mirrors `06_evaluate.py` exactly:

* **MCE-IRL** — `greedy_route`: soft value iteration for this (goal, season,
  vessel), then greedy argmax over the time-indexed soft-VI policy.
* **PEMIRL** — `generate_route_pemirl`: $z$ inferred from the vessel's support
  set (`infer_context_support(..., fixed=True)`), deterministic rollout of the
  generator policy in `GCRLNavEnv`.
* **Transfer agent** — `generate_route_policy` with the *same* $z$: the
  released `ppo_on_pemirl` policy rolled out deterministically.

All rollouts share the env horizon of 512 steps; a decode that never reaches
the goal is truncated there.

In [ ]:
METHODS = ["mce_irl", "pemirl", "transfer"]
routes, rows = [], []
for i, (ep, km) in enumerate(showcase):
    task = Task(start=int(ep.states[0]), goal=int(ep.goal), year=ep.year,
                month=ep.month, mmsi=ep.mmsi, category=ep.category)
    sup = support_map[ep.mmsi][0]                      # support set only
    z = pem.infer_context_support(
        [_episode_tensors(e, fb) for e in sup], fixed=True)

    t0 = time.time()
    decoded = {
        "real": list(ep.states),
        "mce_irl": mce.greedy_route(task.start, task.goal, ep.year, ep.month,
                                    ep.mmsi, horizon=env.max_horizon),
        "pemirl": generate_route_pemirl(pem, env, task, sup, fb),
        "transfer": generate_route_policy(transfer, env, task, z=z,
                                          deterministic=True, device=DEVICE),
    }
    routes.append({"ep": ep, "real_km": km, **decoded})
    for m in METHODS:
        met = route_metrics(mdp, decoded[m], decoded["real"])
        rows.append({"pair": i, "method": m, "real_km": round(km),
                     "decoded_steps": len(decoded[m]), **met})
    print(f"pair {i}: decoded in {time.time()-t0:5.1f}s | "
          + " | ".join(f"{m} H={route_metrics(mdp, decoded[m], decoded['real'])['hausdorff_km']:6.1f} km"
                       for m in METHODS))
df = pd.DataFrame(rows)

## Gallery figure

One map panel per O–D pair: **real AIS route (black)**, **MCE-IRL (blue)**,
**PEMIRL (red)**, **transfer agent (green)**; start = black dot, goal = gold
star. The per-panel box reports each method's Hausdorff distance to the real
route (an asterisk marks decodes that actually reached the goal). Light-blue
dots are the 14,206 navigable H3 res-6 water cells of the graph — if Natural
Earth land polygons cannot be loaded (offline), this scatter still implies the
coastline. Panel extents are anchored on the real route; heavily wandering
decodes may leave the frame (their Hausdorff value tells that story).

In [ ]:
import matplotlib as mpl

# compat shim: cartopy 0.24 calls mpl.RcParams._get(), added in matplotlib 3.7;
# this venv pins matplotlib 3.6.0, where _get is equivalent to plain dict access
if not hasattr(mpl.RcParams, "_get"):
    mpl.RcParams._get = dict.__getitem__

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# -- preflight: are the Natural Earth polygons available (cached or online)? --
try:
    from cartopy.io import shapereader
    shapereader.natural_earth(resolution="50m", category="physical", name="land")
    shapereader.natural_earth(resolution="50m", category="physical",
                              name="coastline")
    NE_OK = True
except Exception as e:                      # offline + no cache
    NE_OK = False
    print(f"Natural Earth unavailable ({e!r}); falling back to water-cell "
          "scatter background")

PC = ccrs.PlateCarree()
cell_lat, cell_lon = mdp.latlng[:, 0], mdp.latlng[:, 1]
STYLE = {"real":     dict(color="black",     lw=2.4, zorder=6, label="Real (AIS)"),
         "mce_irl":  dict(color="tab:blue",  lw=1.5, zorder=5, label="MCE-IRL"),
         "pemirl":   dict(color="tab:red",   lw=1.2, zorder=4, alpha=0.85,
                          label="PEMIRL decode"),
         "transfer": dict(color="tab:green", lw=1.6, zorder=5, alpha=0.95,
                          label="Transfer agent (PPO on frozen PEMIRL reward)")}
ABBR = {"mce_irl": "MCE", "pemirl": "PEM", "transfer": "TRF"}


def panel_extent(r, pad=0.30, cap_deg=14.0, aspect=1.6):
    """Extent anchored on the real route; include decodes up to a cap, then
    expand the narrow dimension so every panel has the same effective
    width:height ratio (lon spans scaled by cos(lat) at Arctic latitudes)."""
    R = mdp.latlng[np.asarray(r["real"], int)]
    c_la, c_lo = R[:, 0].mean(), R[:, 1].mean()
    A = mdp.latlng[np.asarray(sum((r[m] for m in METHODS), r["real"]), int)]
    lo0 = max(A[:, 1].min(), c_lo - cap_deg); lo1 = min(A[:, 1].max(), c_lo + cap_deg)
    la0 = max(A[:, 0].min(), c_la - cap_deg / 2); la1 = min(A[:, 0].max(), c_la + cap_deg / 2)
    lo0 = min(lo0, R[:, 1].min()); lo1 = max(lo1, R[:, 1].max())
    la0 = min(la0, R[:, 0].min()); la1 = max(la1, R[:, 0].max())
    dlo = max(lo1 - lo0, 1.5) * (1 + 2 * pad)
    dla = max(la1 - la0, 0.8) * (1 + 2 * pad)
    # equalize effective aspect (km-true) across panels
    coslat = np.cos(np.radians((la0 + la1) / 2))
    if dlo * coslat / dla < aspect:
        dlo = aspect * dla / coslat
    else:
        dla = dlo * coslat / aspect
    c_lo, c_la = (lo0 + lo1) / 2, (la0 + la1) / 2
    return [c_lo - dlo / 2, c_lo + dlo / 2, c_la - dla / 2, c_la + dla / 2]


fig = plt.figure(figsize=(17, 10.5))
for i, r in enumerate(routes):
    ext = panel_extent(r)
    proj = ccrs.Stereographic(central_longitude=(ext[0] + ext[1]) / 2,
                              central_latitude=(ext[2] + ext[3]) / 2)
    ax = fig.add_subplot(2, 3, i + 1, projection=proj)
    ax.set_extent(ext, crs=PC)
    if NE_OK:
        ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#ece7da",
                       edgecolor="none", zorder=0)
        ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.5,
                       color="0.45", zorder=1)
    ax.scatter(cell_lon, cell_lat, s=0.5, color="#c8dff0", transform=PC,
               zorder=1.5, rasterized=True)          # navigable water cells
    ax.gridlines(lw=0.3, alpha=0.35, draw_labels=False)

    for m in ["pemirl", "mce_irl", "transfer", "real"]:   # real drawn on top
        P = mdp.latlng[np.asarray(r[m], int)]
        ax.plot(P[:, 1], P[:, 0], transform=PC, **STYLE[m])
    s_la, s_lo = mdp.latlng[int(r["real"][0])]
    g_la, g_lo = mdp.latlng[int(r["real"][-1])]
    ax.plot(s_lo, s_la, "o", color="black", ms=8, mec="white", mew=1.2,
            transform=PC, zorder=8)
    ax.plot(g_lo, g_la, "*", color="gold", ms=17, mec="black", mew=0.9,
            transform=PC, zorder=8)

    dm = {row["method"]: row for row in rows if row["pair"] == i}
    txt = "Hausdorff vs real:\n" + "\n".join(
        f"{ABBR[m]} {dm[m]['hausdorff_km']:6.1f} km"
        + ("*" if dm[m]["reached_goal"] else "") for m in METHODS)
    ax.text(0.02, 0.02, txt, transform=ax.transAxes, fontsize=8.5,
            family="monospace", va="bottom", ha="left",
            bbox=dict(fc="white", alpha=0.85, ec="0.6", lw=0.6), zorder=9)
    ep = r["ep"]
    ax.set_title(f"Pair {i} — {ep.category}, {ep.year}-{ep.month:02d}, "
                 f"real route {r['real_km']:.0f} km", fontsize=10.5)

handles = [plt.Line2D([], [], **{k: v for k, v in STYLE[m].items()
                                 if k in ("color", "lw", "alpha", "label")})
           for m in ["real", "mce_irl", "pemirl", "transfer"]]
handles += [plt.Line2D([], [], marker="o", color="black", ls="", ms=8,
                       mec="white", label="Start"),
            plt.Line2D([], [], marker="*", color="gold", ls="", ms=14,
                       mec="black", label="Goal"),
            plt.Line2D([], [], ls="", label="* = decode reached the goal")]
fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=9.5,
           frameon=False, bbox_to_anchor=(0.5, -0.005))
fig.suptitle("Decoded routes vs. real Arctic voyages — six held-out test O–D "
             "pairs (vessel-disjoint split, PEMIRL/transfer conditioned on "
             "per-vessel support sets)", fontsize=13, y=0.995)
fig.tight_layout(rect=(0, 0.035, 1, 0.975))
fig.savefig("notebooks/figures/fig02_route_gallery.png", dpi=300,
            bbox_inches="tight")
plt.show()
print("saved notebooks/figures/fig02_route_gallery.png")

In [ ]:
# companion figure: Hausdorff by pair and method (log scale)
fig, ax = plt.subplots(figsize=(8.5, 4))
w, x = 0.26, np.arange(len(routes))
for j, m in enumerate(METHODS):
    v = [dm["hausdorff_km"] for p in x
         for dm in [next(r for r in rows if r["pair"] == p and r["method"] == m)]]
    reached = [next(r for r in rows if r["pair"] == p and r["method"] == m)
               ["reached_goal"] for p in x]
    bars = ax.bar(x + (j - 1) * w, v, w, color=STYLE[m]["color"],
                  label=STYLE[m]["label"], alpha=0.9)
    for b, rc in zip(bars, reached):
        if rc:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() * 1.06, "*",
                    ha="center", fontsize=13, fontweight="bold")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels([f"pair {p}\n({r['real_km']:.0f} km)"
                    for p, r in enumerate(routes)], fontsize=9)
ax.set_ylabel("Hausdorff distance to real route (km, log)")
ax.set_title("Route fidelity per showcase pair (* = decode reached the goal)",
             fontsize=11)
ax.legend(fontsize=9, frameon=False)
ax.grid(axis="y", lw=0.3, alpha=0.4)
fig.tight_layout()
fig.savefig("notebooks/figures/fig02_hausdorff_bars.png", dpi=300,
            bbox_inches="tight")
plt.show()
print("saved notebooks/figures/fig02_hausdorff_bars.png")

## Summary table

In [ ]:
pd.set_option("display.width", 200)
tbl = df.pivot(index="pair", columns="method",
               values=["hausdorff_km", "length_ratio", "reached_goal"])
tbl = tbl.reindex(columns=METHODS, level=1).round(2)
tbl.insert(0, ("real_km", ""), [round(r["real_km"]) for r in routes])
display(tbl)
print("\nMeans over the 6 showcase pairs:")
display(df.groupby("method")[["hausdorff_km", "length_ratio",
                              "reached_goal"]].mean().round(2).reindex(METHODS))

## Discussion

**The dissociation is plainly visible.** MCE-IRL (blue) tracks the real
corridor on every panel — mean Hausdorff ≈ 39 km, and on five of six pairs it
is the closest or near-closest decode — yet it *never terminates at the goal*
(reached_goal = 0 on all six pairs, consistent with 0/200 in the reference
evaluation): the greedy soft-VI decode stalls along the corridor and burns the
512-step horizon, giving length ratios below 1. PEMIRL (red), despite winning
the per-decision log-likelihood by ~10.5%, decodes an order of magnitude worse
(mean Hausdorff ≈ 213 km, length ratios up to ~28): its reactive step-by-step
generator compounds small errors into long wandering loops, exactly the
failure the paper's route-fidelity metrics report in aggregate.

**The transfer agent is the new evidence, and it cuts both ways —
informatively.** On the two shortest pairs (127 and 209 km real routes) the
fresh PPO agent trained *only* on the frozen PEMIRL reward is the **only
method that actually completes the voyage**: it reaches the goal in 17 and 30
steps with near-expert geometry (Hausdorff 12.2 and 28.0 km, length ratios
0.89 and 0.87 — on pair 1 it beats even MCE-IRL's 39.2 km). This is the
strongest possible sanity check on the learned reward: a policy that never saw
an expert action recovers direct, goal-reaching, corridor-following routes
from the reward signal alone. On the longer transits, however, the transfer
agent degrades toward PEMIRL-like wandering (pairs 3 and 5: Hausdorff ≈ 424
and 665 km, no goal reached), indicating the learned reward's useful gradient
is strongest near-goal / short-horizon and does not yet carry a 1,000+ km
route to completion within the 512-step horizon.

**Takeaway for the paper.** Per-decision likelihood (PEMIRL's win) measures
*local move ranking*; route decoding measures *global plan consistency*
(MCE-IRL's win, by virtue of explicit soft-VI planning); and reward transfer
measures *whether the learned reward is a usable training signal*. The
transfer agent's goal-reaching, near-expert routes on short/medium O–D pairs —
where PEMIRL's own generator fails — show the context-conditioned reward
contains genuinely usable navigation preferences, while its long-range
failures delimit how far that signal currently reaches.